In [ ]:
import helpers.i2c_gui2_helpers as helpers
import datetime
import numpy as np

In [ ]:
chip_names = ["ET2FFF_EP_NH2"]
# chip_names = ["ET2p03_16x32_HPK2_left"]


# 'The port name the USB-ISS module is connected to. Default: /dev/ttyACM0'
port = "/dev/ttyACM0"
chip_addresses = [0x60]
ws_addresses = [None]

In [ ]:
i2c_conn = helpers.i2c_connection(port,chip_addresses,ws_addresses,chip_names)

In [ ]:
# Calibrate PLL
for chip_address in chip_addresses[:]:
    i2c_conn.calibratePLL(chip_address, chip=None)
# Calibrate FC for all I2C
for chip_address in chip_addresses[:]:
    i2c_conn.asyResetGlobalReadout(chip_address, chip=None)
    i2c_conn.asyAlignFastcommand(chip_address, chip=None)

In [ ]:
i2c_conn.config_chips(
    do_pixel_check=False,
    do_basic_peripheral_register_check=False, ### Need to re-visit
    do_disable_all_pixels=False,
    do_auto_calibration=False,
    do_disable_and_calibration=True,
    do_prepare_ws_testing=False
)

In [ ]:
# i2c_conn.get_bl_nw_map()
# bls = i2c_conn.get_bl_nw_map()

In [ ]:
now = datetime.datetime.now().isoformat(sep=' ', timespec='seconds')
i2c_conn.save_baselines(save_notes=f'{now} 140V')

In [ ]:
qinj_test = False

if (qinj_test):
    row_list = [14, 14, 15, 15]
    col_list = [6, 9, 6, 9] 
    scan_list = list(zip(row_list, col_list))

else:
    col_list, row_list = np.meshgrid(np.arange(16),np.arange(16))
    scan_list = list(zip(row_list.flatten(),col_list.flatten()))

    # row_list = [15] * 8
    # col_list = [13, 12, 11, 10, 9, 8, 7, 6] 
    # scan_list = list(zip(row_list, col_list))

print(scan_list)

In [ ]:
bypass_on = True

if (qinj_test):
    i2c_conn.enable_select_pixels_in_chips(scan_list, Qsel=30, QInjEn=True, Bypass_THCal=bypass_on, power_mode='high', verbose=False)
else:
    i2c_conn.enable_select_pixels_in_chips(scan_list, Qsel=5, QInjEn=False, Bypass_THCal=bypass_on, power_mode='high', verbose=False)

In [ ]:
offset = 15
# offset = 0x14 ## WB Cosmic run

if (qinj_test):
    offset = 10

for chip_address in chip_addresses:
    chip = i2c_conn.get_chip_i2c_connection(chip_address)
    i2c_conn.set_chip_offsets(chip_address, pixel_list=scan_list, offset=offset, chip=chip, verbose=False)
    del chip

In [ ]:
# # 
# for chip_address in chip_addresses:
#     i2c_conn
#     pass

In [ ]:
i2c_conn.config_fc_data_delay(chip_addresses[0], fc_clk_delay=1, fc_data_delay=1)